# Vérification du téléchargement des infrastructures OpenStreetMap

Ce notebook vérifie que `download.osm.download_osm_infrastructures` interroge correctement OpenStreetMap (Overpass API via osmnx) sur l'emprise d'une entité du CSV, écrit chaque catégorie en GeoPackage dans `data/vector/infrastructures/`, et affiche le résultat sur une carte.

In [ ]:
import sys
from pathlib import Path

# GeoDataEngine/ (racine du package core/) et son parent (racine du repo, contient data/)
project_root = Path.cwd().parent
repo_root = project_root.parent
sys.path.insert(0, str(project_root))

from core.config import load_entities
from download.osm import download_osm_infrastructures

## 1. Entité testée (issue du CSV)

In [ ]:
csv_path = repo_root / "data" / "configuration" / "entites_exemple.csv"
entity = load_entities(csv_path)[0]
entity

## 2. Téléchargement des infrastructures OSM sur la bbox de l'entité

In [ ]:
written = download_osm_infrastructures(entity.bbox)
written

## 3. Relecture des GeoPackage écrits

In [ ]:
import geopandas as gpd

layers = {name: gpd.read_file(path) for name, path in written.items()}
for name, gdf in layers.items():
    print(name, "->", len(gdf), "entites")

## 4. Affichage sur une carte `leafmap`

Reprojection en EPSG:4326 uniquement pour l'affichage. Backend `folium` (HTML statique), plus fiable pour le rendu des widgets dans VS Code.

In [ ]:
import leafmap.foliumap as leafmap

colors = {
    "ecoles": "blue",
    "restauration": "orange",
    "commerces": "red",
    "sante": "green",
    "services_publics": "purple",
    "sport_loisirs": "darkgreen",
}

m = leafmap.Map()

last_gdf_wgs84 = None
for name, gdf in layers.items():
    if gdf.empty:
        continue
    gdf_wgs84 = gdf.to_crs(epsg=4326)
    last_gdf_wgs84 = gdf_wgs84
    m.add_gdf(
        gdf_wgs84,
        layer_name=name,
        style={"color": colors.get(name, "gray")},
        info_mode="on_click",
    )

if last_gdf_wgs84 is not None:
    m.zoom_to_gdf(last_gdf_wgs84)

m